In [7]:
import pandas as pd
import networkx as nx

prereqs = pd.read_csv(
    "../data/processed/prerequisites_fall_2026.csv"
)

prereqs["course"] = (
    prereqs["subject"] + " " + prereqs["course_number"].astype(str)
)

prereqs["prerequisite"] = (
    prereqs["prereq_subject"] + " " + prereqs["prereq_number"].astype(str)
)

G = nx.DiGraph()

for _, row in prereqs.iterrows():
    G.add_edge(
        row["prerequisite"],
        row["course"]
    )

In [8]:
completed_courses = {
    "DATA C8",
    "COMPSCI 61A",
    "MATH 1A",
    "MATH 1B"
}

In [9]:
def get_prerequisites(course):
    if course not in G:
        return set()

    return set(G.predecessors(course))

In [10]:
def is_eligible(course, completed_courses):
    prereqs = get_prerequisites(course)

    return prereqs.issubset(completed_courses)

In [11]:
get_prerequisites("AEROENG 10")

{'COMPSCI 61A', 'ENGIN 7', 'MATH 51', 'MATH 52', 'MATH 53', 'PHYSICS 7A'}

In [12]:
def missing_prerequisites(course, completed_courses):
    return get_prerequisites(course) - completed_courses

In [13]:
missing_prerequisites("AEROENG 10", completed_courses)

{'ENGIN 7', 'MATH 51', 'MATH 52', 'MATH 53', 'PHYSICS 7A'}

In [14]:
is_eligible("AEROENG 10", completed_courses)

False

In [16]:
courses = pd.read_csv(
    "../data/processed/recommendable_courses_fall_2026.csv"
)

courses["course"] = (
    courses["subject"] + " " + courses["course_number"].astype(str)
)

courses["missing_prereqs"] = courses["course"].apply(
    lambda course: missing_prerequisites(course, completed_courses)
)

courses["eligible"] = courses["missing_prereqs"].apply(
    lambda missing: len(missing) == 0
)

In [17]:
courses["eligible"].value_counts()

eligible
True     1725
False     394
Name: count, dtype: int64

In [18]:
eligible_courses = courses[
    courses["eligible"]
].copy()

eligible_courses.shape

(1725, 11)

In [19]:
eligible_courses[
    ["course", "title", "requirements"]
].sample(20, random_state=42)

,course,title,requirements
445,CYPLAN 124,Sustainable Mobility,NaN
786,ETHSTD 190,Advanced Seminar in Comparative Ethnic Studies,Consent of instructor.
383,COMLIT 100D,Introduction to Comparative Literature,NaN
1875,SOCIOL 5,Evaluation of Evidence,NaN
594,ENGIN 183B,Berkeley Method of Entrepreneurship Bootcamp,NaN
1638,POLSCI 111AC,The Politics of Displacement,NaN
27,AFRICAM 15A,Advanced Swahili,NaN
1044,INDONES 100A,Intermediate Indonesian,1A-1B.
1779,RHETOR 123,Rhetoric of Performance,"Any 1A-1B sequence, upper divison standing, an..."
1588,PHYSICS 24,Freshman Seminars,NaN


In [20]:
courses.loc[
    ~courses["eligible"],
    ["course", "title", "missing_prereqs"]
].head(20)

,course,title,missing_prereqs
1,AEROENG 10,Introduction to Aerospace Engineering Design,"{MATH 51, ENGIN 7, MATH 52, MATH 53, PHYSICS 7A}"
2,AEROENG 100,Aerospace Capstone,"{MECENG 103, MECENG 104, MECENG 132, MECENG 106}"
4,AEROENG C124,Materials for Extreme Environments,{ENGIN 40}
7,MECENG C162,Introduction to Flight Mechanics,"{MATH 52, PHYSICS 7A}"
66,ANTHRO 106,Primate Behavior,{BIOLOGY 32}
72,ANTHRO 127A,Bioarchaeology: Introduction to Skeletal Biolo...,{BIOLOGY 1B}
78,ANTHRO 150,Utopia: Art and Power in Modern Times,{ANTHRO 3}
91,ARABIC 100A,Advanced Arabic,"{ARABIC 20B, ARABIC 30}"
92,ARABIC 104A,Modern Arabic Prose,"{ARABIC 20B, ARABIC 30}"
93,ARABIC 1A,Elementary Arabic,"{ARABIC 1B, ARABIC 1A}"


### Eligibility Observations
- For the example student profile, 1,725 of 2,119 recommendable courses have no missing parsed course prerequisites.
- 394 courses are blocked by one or more explicit course prerequisites. 
- eligible=True only means that no parsed course prerequisites are missing; non-course requirements such as instructor consent, auditions, GPA thresholds, or standing restrictions are not yet enforced.